# Advanced OOP Patterns

**OOP-in-Python — Advanced**  
Metaclasses, descriptors, `__slots__`, dataclasses, Enums, and more.

## Overview

| Concept | Purpose | When to Use |
|---|---|---|
| **Metaclasses** | Classes of classes — control class creation | ORMs, frameworks, APIs |
| **Descriptors** | Reusable attribute behavior | Validation, computed attrs |
| **`__slots__`** | Memory optimization | Many instances (millions) |
| **Dataclasses** | Auto-generated boilerplate | Data containers |
| **Enums** | Named constants | Fixed sets of values |
| **NamedTuple** | Immutable, lightweight data | Simple struct-like objects |

## 1. Metaclasses

A **metaclass** is a class of a class. `type` is the default metaclass. Metaclasses control how classes are **created** and **behave**.

In [ ]:
# Everything is an object — even classes
class MyClass:
    pass

obj = MyClass()
print(type(obj))       # <class '__main__.MyClass'>
print(type(MyClass))   # <class 'type'> — type is the metaclass!
print(type(type))      # <class 'type'> — type is its own metaclass

In [ ]:
# Custom metaclass: auto-adds a 'created_at' attribute to every class
from datetime import datetime

class TimestampMeta(type):
    def __new__(mcs, name: str, bases: tuple, namespace: dict):
        # Called when a class using this metaclass is defined
        namespace['created_at'] = datetime.now().isoformat()
        return super().__new__(mcs, name, bases, namespace)

class MyService(metaclass=TimestampMeta):
    def serve(self) -> str:
        return "Serving..."

print(MyService.created_at)  # Auto-added!
print(MyService().serve())

In [ ]:
# Metaclass for automatic method registration
class PluginRegistry(type):
    plugins = {}

    def __new__(mcs, name, bases, namespace):
        cls = super().__new__(mcs, name, bases, namespace)
        if hasattr(cls, 'plugin_name'):
            mcs.plugins[cls.plugin_name] = cls
        return cls

class BasePlugin(metaclass=PluginRegistry):
    plugin_name = None

class EmailPlugin(BasePlugin):
    plugin_name = "email"
    def send(self, msg: str):
        return f"Email: {msg}"

class SMSPlugin(BasePlugin):
    plugin_name = "sms"
    def send(self, msg: str):
        return f"SMS: {msg}"

print(PluginRegistry.plugins)
# {'email': <class 'EmailPlugin'>, 'sms': <class 'SMSPlugin'>}

## 2. Descriptors

Descriptors let you define **reusable property logic** via `__get__`, `__set__`, `__delete__`.

In [ ]:
class ValidatedAttribute:
    """Descriptor that validates attribute values."""
    def __init__(self, validator):
        self.validator = validator
        self.data = {}  # Per-instance storage

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return self.data.get(id(obj), self.validator.default)

    def __set__(self, obj, value):
        if not self.validator.is_valid(value):
            raise ValueError(f"Invalid value: {value}")
        self.data[id(obj)] = value

    def __delete__(self, obj):
        del self.data[id(obj)]

class PositiveNumber:
    default = 0
    @staticmethod
    def is_valid(value) -> bool:
        return isinstance(value, (int, float)) and value >= 0

class NonEmptyString:
    default = ""
    @staticmethod
    def is_valid(value) -> bool:
        return isinstance(value, str) and len(value.strip()) > 0

class Person:
    name = ValidatedAttribute(NonEmptyString())
    age = ValidatedAttribute(PositiveNumber())

p = Person()
p.name = "Alice"  # OK
p.age = 30         # OK
# p.age = -5      # ValueError!
# p.name = ""    # ValueError!
print(f"{p.name} is {p.age} years old")

## 3. `__slots__` for Memory Optimization

By default, Python stores instance attributes in a `__dict__` (dictionary), which is memory-heavy. `__slots__` tells Python to use a **fixed-size array** instead.

In [ ]:
import sys

class Point:
    __slots__ = ('x', 'y')  # Only these attributes allowed

    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    def __repr__(self) -> str:
        return f"Point({self.x}, {self.y})"

p = Point(3, 4)
print(p.x, p.y)  # 3, 4
# p.z = 5  # AttributeError: 'Point' has no attribute 'z'
# print(p.__dict__)  # AttributeError: no __dict__

# Memory comparison
class PointNoSlots:
    def __init__(self, x, y):
        self.x = x
        self.y = y

p1 = Point(1, 2)
p2 = PointNoSlots(1, 2)
print(f"With __slots__: {sys.getsizeof(p1)} bytes")
print(f"Without __slots__: {sys.getsizeof(p2)} bytes")
# For millions of instances, __slots__ saves significant memory

## 4. `__init_subclass__`

Hook that runs whenever a subclass is created — useful for registration and validation.

In [ ]:
class Base:
    _registry = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        # Called when any subclass of Base is defined
        Base._registry[cls.__name__] = cls
        print(f"Registered: {cls.__name__}")

class Cat(Base):
    pass

class Dog(Base):
    pass

print(Base._registry)  # {'Cat': <class 'Cat'>, 'Dog': <class 'Dog'>}

## 5. Dataclasses

Python 3.7+ `dataclasses` auto-generate `__init__`, `__repr__`, `__eq__`, `__hash__`, and more.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Product:
    """Product with auto-generated __init__, __repr__, __eq__."""
    name: str
    price: float
    quantity: int = 0
    tags: list = field(default_factory=list)  # Mutable default

    @property
    def total_value(self) -> float:
        return self.price * self.quantity

p1 = Product("Widget", 19.99, 100)
p2 = Product("Widget", 19.99, 100)
print(p1)                 # Product(name='Widget', price=19.99, quantity=100, tags=[])
print(p1 == p2)          # True (auto __eq__)
print(p1.total_value)    # 1999.0

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class Point:
    """Immutable point — acts like a value object."""
    x: float
    y: float

pt = Point(3, 4)
print(pt.x, pt.y)  # 3, 4
# pt.x = 5  # FrozenInstanceError: cannot assign to field

In [ ]:
from dataclasses import dataclass, field

@dataclass
class User:
    name: str
    email: str
    _id: int = field(init=False, repr=False)  # Computed, not in init/repr

    def __post_init__(self):
        # Called after auto-generated __init__
        self._id = hash(self.email) & 0xFFFFFFFF

u = User("Alice", "alice@example.com")
print(f"ID: {u._id}, Repr: {u}")

## 6. Enum Class

Enums provide **named constants** with type safety.

In [ ]:
from enum import Enum, auto

class Color(Enum):
    RED = 1
    GREEN = 2
    BLUE = 3

class Status(Enum):
    PENDING = auto()
    APPROVED = auto()
    REJECTED = auto()

print(Color.RED)                 # Color.RED
print(Color.RED.name)            # RED
print(Color.RED.value)           # 1
print(list(Color))               # [<Color.RED: 1>, <Color.GREEN: 2>, <Color.BLUE: 3>]
print(Color(2))                  # Color.GREEN (reverse lookup)
print(isinstance(Color.RED, Color))  # True

def process_order(status: Status):
    if status == Status.APPROVED:
        return "Processing..."
    elif status == Status.PENDING:
        return "Waiting..."

print(process_order(Status.APPROVED))  # Processing...

## 7. NamedTuple

Lightweight, immutable data containers with named fields.

In [ ]:
from typing import NamedTuple

class Address(NamedTuple):
    street: str
    city: str
    zip_code: str

addr = Address("123 Main St", "Portland", "97201")
print(addr.street)    # 123 Main St
print(addr[0])        # 123 Main St (tuple-like)
print(addr)           # Address(street='123 Main St', city='Portland', zip_code='97201')
street, city, zipcode = addr  # Unpacking
print(f"City: {city}")

---
## Problem 1: Validator Descriptor

**Difficulty: Hard**

Build a reusable `Validator` descriptor that supports multiple validation rules.

In [ ]:
from typing import Callable, Any

class Validator:
    def __init__(self, *validators: Callable[[Any], bool], default=None):
        self.validators = validators
        self.default = default
        self.data = {}  # id(obj) -> value

    def __set_name__(self, owner, name):
        self._name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return self.data.get(id(obj), self.default)

    def __set__(self, obj, value):
        for validator in self.validators:
            if not validator(value):
                raise ValueError(f"Validation failed for {self._name}: {value!r}")
        self.data[id(obj)] = value

    def __delete__(self, obj):
        del self.data[id(obj)]

# Validation functions
def is_non_empty(s: str) -> bool:
    return isinstance(s, str) and len(s.strip()) > 0

def is_positive(n: (int, float)) -> bool:
    return isinstance(n, (int, float)) and n > 0

def is_email(s: str) -> bool:
    return isinstance(s, str) and '@' in s and '.' in s.split('@')[-1]

def is_age(n: int) -> bool:
    return isinstance(n, int) and 0 <= n <= 150

class Employee:
    name = Validator(is_non_empty, default="Unknown")
    email = Validator(is_non_empty, is_email)
    age = Validator(is_age, default=18)
    salary = Validator(is_positive)

e = Employee()
e.name = "Alice"
e.email = "alice@company.com"
e.age = 30
e.salary = 75000
print(f"{e.name}, {e.email}, {e.age}, ${e.salary}")

try:
    e.email = "invalid"
except ValueError as ex:
    print(f"Caught: {ex}")

## Problem 2: Custom ORM-like Field System

**Difficulty: Hard**

Build a mini ORM field system using descriptors and metaclasses.

In [ ]:
from typing import Any

class Field:
    """Base descriptor for model fields."""
    def __init__(self, field_type: type, default=None, nullable: bool = True):
        self.field_type = field_type
        self.default = default
        self.nullable = nullable
        self.name = None

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name, self.default)

    def __set__(self, obj, value):
        if value is None and not self.nullable:
            raise ValueError(f"{self.name} cannot be null")
        if value is not None and not isinstance(value, self.field_type):
            raise TypeError(f"{self.name} must be {self.field_type.__name__}, got {type(value).__name__}")
        obj.__dict__[self.name] = value

class IntegerField(Field):
    def __init__(self, default=0, nullable=False, min_value=None, max_value=None):
        super().__init__(int, default, nullable)
        self.min_value = min_value
        self.max_value = max_value

    def __set__(self, obj, value):
        super().__set__(obj, value)
        if value is not None:
            if self.min_value is not None and value < self.min_value:
                raise ValueError(f"{self.name} min is {self.min_value}")
            if self.max_value is not None and value > self.max_value:
                raise ValueError(f"{self.name} max is {self.max_value}")

class CharField(Field):
    def __init__(self, max_length: int = 255, default="", nullable=False):
        super().__init__(str, default, nullable)
        self.max_length = max_length

    def __set__(self, obj, value):
        super().__set__(obj, value)
        if value is not None and len(value) > self.max_length:
            raise ValueError(f"{self.name} exceeds max length {self.max_length}")

class ModelMeta(type):
    def __new__(mcs, name, bases, namespace):
        cls = super().__new__(mcs, name, bases, namespace)
        cls._fields = {}
        for attr_name, attr_value in namespace.items():
            if isinstance(attr_value, Field):
                cls._fields[attr_name] = attr_value
        return cls

class Model(metaclass=ModelMeta):
    _table = None

    def __init__(self, **kwargs):
        for field_name, field in self._fields.items():
            value = kwargs.get(field_name, field.default)
            setattr(self, field_name, value)

    def to_dict(self) -> dict:
        return {name: getattr(self, name) for name in self._fields}

    def validate(self):
        for name, field in self._fields.items():
            value = getattr(self, name)
            setattr(self, name, value)  # Triggers validation

class User(Model):
    _table = "users"
    id = IntegerField(min_value=1)
    name = CharField(max_length=100)
    email = CharField(max_length=255)
    age = IntegerField(min_value=0, max_value=150, nullable=True)

# Usage
u = User(id=1, name="Alice", email="alice@example.com", age=30)
print(u.to_dict())
print(User._fields)  # {'id': IntegerField, 'name': CharField, ...}

try:
    User(name="Bob", email="bob@example.com", id=-1, age=25)
except ValueError as ex:
    print(f"Caught: {ex}")

## Key Takeaways

- **Metaclasses**: control class creation — use for frameworks, ORMs, registries.
- **Descriptors**: reusable attribute logic — the engine behind `@property`, `@classmethod`, `@staticmethod`.
- **`__slots__`**: memory optimization — skip `__dict__` for fixed-attribute classes.
- **`__init_subclass__`**: hook into subclass creation without metaclasses.
- **Dataclasses**: auto-generate boilerplate — great for data containers.
- **Enums**: type-safe named constants — `auto()` for auto-values.
- **NamedTuple**: lightweight immutable data — tuple with named fields.
- These advanced patterns are the building blocks of Python's standard library and major frameworks.